# Notebook 07 (Part A — Tabular): Hybrid Anomaly Detection Framework

Addresses RQ2/H2 from the approved Project Approval Form: *"Can a hybrid anomaly detection
framework improve defect detection performance under conditions of limited labelled defect
data compared with individual deep learning and classical anomaly detection methods?"*

**Design choice (stated explicitly, not hidden):** this notebook fuses the best-performing
classical method (Isolation Forest) with the best-performing deep learning method
(Feedforward Autoencoder) for the tabular dataset, to test genuine cross-paradigm
complementarity, consistent with the reconstruction-based / boundary-based / classical
framework used throughout this study.

**Three fusion strategies are tested** (per the approval form's own wording that the fusion
design "will be developed during the research rather than predetermined in advance"):
1. Simple average of normalized scores
2. Weighted average, with the weight selected on the **validation set only** (no test-set leakage)
3. Maximum of normalized scores

Scores are normalized (min-max) using statistics fit **only on the normal training data**,
consistent with the one-class protocol used throughout this study. The same 10-seed,
3-threshold-condition protocol from Notebook 04/06 is used, with FPR included.

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                              precision_score, recall_score, confusion_matrix)
from scipy.stats import wilcoxon
import pickle, warnings
warnings.filterwarnings('ignore')

BASE = r'C:\Users\Administrator\Documents\kf7029-w25039720'

X_train_normal = np.load(fr'{BASE}\datasets\usdot-pipeline-accidents\X_train_normal.npy')
X_val          = np.load(fr'{BASE}\datasets\usdot-pipeline-accidents\X_val.npy')
X_test         = np.load(fr'{BASE}\datasets\usdot-pipeline-accidents\X_test.npy')
y_val          = np.load(fr'{BASE}\datasets\usdot-pipeline-accidents\y_val.npy')
y_test         = np.load(fr'{BASE}\datasets\usdot-pipeline-accidents\y_test.npy')

print('Tabular data loaded:', X_train_normal.shape, X_val.shape, X_test.shape)

Tabular data loaded: (1541, 72) (280, 72) (559, 72)


In [2]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, 32), nn.ReLU(), nn.Linear(32,16), nn.ReLU(), nn.Linear(16,8))
        self.decoder = nn.Sequential(nn.Linear(8,16), nn.ReLU(), nn.Linear(16,32), nn.ReLU(), nn.Linear(32, input_dim))
    def forward(self, x):
        return self.decoder(self.encoder(x))

def min_max_fit(train_scores):
    return train_scores.min(), train_scores.max()

def min_max_apply(scores, lo, hi):
    return (scores - lo) / (hi - lo + 1e-12)

def threshold_unsupervised(scores, anomaly_ratio=0.21):
    return np.percentile(scores, 100 * (1 - anomaly_ratio))

def threshold_label_assisted(val_scores, y_val, defect_pct, rng):
    defect_indices = np.where(y_val == 1)[0].copy()
    n_use = max(1, int(len(defect_indices) * defect_pct))
    rng.shuffle(defect_indices)
    used_defect_idx = defect_indices[:n_use]
    normal_indices = np.where(y_val == 0)[0]
    calib_idx = np.concatenate([normal_indices, used_defect_idx])
    calib_scores = val_scores[calib_idx]
    calib_labels = y_val[calib_idx]
    best_threshold, best_f1 = 0, 0
    for pct in range(50, 99):
        t = np.percentile(calib_scores, pct)
        preds = (calib_scores >= t).astype(int)
        if preds.sum() == 0: continue
        f1 = f1_score(calib_labels, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_threshold = f1, t
    return best_threshold

def collect_metrics(scores_test, scores_val, y_test, y_val, condition, rng):
    if condition == 'unsupervised':
        t = threshold_unsupervised(scores_test)
    elif condition == 'label_5pct':
        t = threshold_label_assisted(scores_val, y_val, 0.05, rng)
    elif condition == 'label_10pct':
        t = threshold_label_assisted(scores_val, y_val, 0.10, rng)
    preds = (scores_test >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds, labels=[0,1]).ravel()
    fpr = fp / (fp + tn) if (fp+tn) > 0 else 0.0
    return {'ROC-AUC': roc_auc_score(y_test, scores_test), 'PR-AUC': average_precision_score(y_test, scores_test),
            'F1': f1_score(y_test, preds, zero_division=0), 'Precision': precision_score(y_test, preds, zero_division=0),
            'Recall': recall_score(y_test, preds, zero_division=0), 'FPR': fpr}

print('Helper functions defined.')

Helper functions defined.


In [3]:
SEEDS = [42, 7, 13, 99, 21, 55, 77, 3, 88, 11]
conditions = ['unsupervised', 'label_5pct', 'label_10pct']
metrics_list = ['ROC-AUC','PR-AUC','F1','Precision','Recall','FPR']
input_dim = X_train_normal.shape[1]

model_names = ['Isolation Forest', 'Feedforward AE', 'Hybrid - Average', 'Hybrid - Weighted', 'Hybrid - Max']
tabular_hybrid_runs = {name: {c: {m: [] for m in metrics_list} for c in conditions} for name in model_names}
selected_weights = []

X_tr_t = torch.FloatTensor(X_train_normal)
X_va_t = torch.FloatTensor(X_val)
X_te_t = torch.FloatTensor(X_test)

for seed_idx, seed in enumerate(SEEDS):
    rng = np.random.RandomState(seed)
    print(f'Run {seed_idx+1}/10 (seed={seed})')

    # Isolation Forest
    iso = IsolationForest(n_estimators=100, contamination=0.21, random_state=seed)
    iso.fit(X_train_normal)
    iso_train = -iso.score_samples(X_train_normal)
    iso_val = -iso.score_samples(X_val)
    iso_test = -iso.score_samples(X_test)

    # Feedforward AE - retrained per seed, matching Notebook 04/06 protocol
    torch.manual_seed(seed)
    ae_model = Autoencoder(input_dim)
    ae_optim = torch.optim.Adam(ae_model.parameters(), lr=0.001)
    crit = nn.MSELoss()
    loader = DataLoader(TensorDataset(X_tr_t, X_tr_t), batch_size=32, shuffle=True)
    for epoch in range(100):
        for bx, by in loader:
            ae_optim.zero_grad()
            crit(ae_model(bx), by).backward()
            ae_optim.step()
    ae_model.eval()
    with torch.no_grad():
        ae_train = torch.mean((X_tr_t - ae_model(X_tr_t))**2, dim=1).numpy()
        ae_val = torch.mean((X_va_t - ae_model(X_va_t))**2, dim=1).numpy()
        ae_test = torch.mean((X_te_t - ae_model(X_te_t))**2, dim=1).numpy()

    # Normalize using train_normal statistics only
    iso_lo, iso_hi = min_max_fit(iso_train)
    ae_lo, ae_hi = min_max_fit(ae_train)
    iso_val_n, iso_test_n = min_max_apply(iso_val, iso_lo, iso_hi), min_max_apply(iso_test, iso_lo, iso_hi)
    ae_val_n, ae_test_n = min_max_apply(ae_val, ae_lo, ae_hi), min_max_apply(ae_test, ae_lo, ae_hi)

    # Fusion 1: simple average
    fus_avg_val, fus_avg_test = (iso_val_n+ae_val_n)/2, (iso_test_n+ae_test_n)/2

    # Fusion 2: weighted average, weight selected on VALIDATION SET via PR-AUC (no test leakage)
    best_w, best_pr = 0.5, -1
    for w in np.arange(0, 1.05, 0.05):
        pr = average_precision_score(y_val, w*iso_val_n + (1-w)*ae_val_n)
        if pr > best_pr:
            best_pr, best_w = pr, w
    selected_weights.append(best_w)
    fus_w_val = best_w*iso_val_n + (1-best_w)*ae_val_n
    fus_w_test = best_w*iso_test_n + (1-best_w)*ae_test_n

    # Fusion 3: max
    fus_max_val, fus_max_test = np.maximum(iso_val_n, ae_val_n), np.maximum(iso_test_n, ae_test_n)

    run_scores = {
        'Isolation Forest': (iso_test, iso_val),
        'Feedforward AE': (ae_test, ae_val),
        'Hybrid - Average': (fus_avg_test, fus_avg_val),
        'Hybrid - Weighted': (fus_w_test, fus_w_val),
        'Hybrid - Max': (fus_max_test, fus_max_val),
    }
    for name, (stest, sval) in run_scores.items():
        for cond in conditions:
            m = collect_metrics(stest, sval, y_test, y_val, cond, rng)
            for k, v in m.items():
                tabular_hybrid_runs[name][cond][k].append(v)

print(f'\nAll 10 runs complete. Selected weights per run (for Isolation Forest side): {[round(w,2) for w in selected_weights]}')
print(f'Mean selected weight: {np.mean(selected_weights):.2f}')

Run 1/10 (seed=42)
Run 2/10 (seed=7)
Run 3/10 (seed=13)
Run 4/10 (seed=99)
Run 5/10 (seed=21)
Run 6/10 (seed=55)
Run 7/10 (seed=77)
Run 8/10 (seed=3)
Run 9/10 (seed=88)
Run 10/10 (seed=11)

All 10 runs complete. Selected weights per run (for Isolation Forest side): [np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0)]
Mean selected weight: 1.00


In [4]:
print('=== TABULAR HYBRID RESULTS (unsupervised condition, mean ± std, 10 runs) ===\n')
rows = []
for name in model_names:
    r = tabular_hybrid_runs[name]['unsupervised']
    rows.append({'Model': name,
        'ROC-AUC': f"{np.mean(r['ROC-AUC']):.4f} ± {np.std(r['ROC-AUC']):.4f}",
        'PR-AUC':  f"{np.mean(r['PR-AUC']):.4f} ± {np.std(r['PR-AUC']):.4f}",
        'F1':      f"{np.mean(r['F1']):.4f} ± {np.std(r['F1']):.4f}",
        'FPR':     f"{np.mean(r['FPR']):.4f} ± {np.std(r['FPR']):.4f}"})
df_hybrid = pd.DataFrame(rows).set_index('Model')
print(df_hybrid.to_string())

with open(fr'{BASE}\results\tabular_hybrid_runs.pkl', 'wb') as f:
    pickle.dump(tabular_hybrid_runs, f)
df_hybrid.to_csv(fr'{BASE}\results\tabular_hybrid_summary.csv')
print('\nSaved: results/tabular_hybrid_runs.pkl and results/tabular_hybrid_summary.csv')

=== TABULAR HYBRID RESULTS (unsupervised condition, mean ± std, 10 runs) ===

                           ROC-AUC           PR-AUC               F1              FPR
Model                                                                                
Isolation Forest   0.5561 ± 0.0220  0.2407 ± 0.0162  0.2610 ± 0.0403  0.1977 ± 0.0108
Feedforward AE     0.4740 ± 0.0260  0.1975 ± 0.0070  0.1500 ± 0.0257  0.2274 ± 0.0069
Hybrid - Average   0.5476 ± 0.0214  0.2326 ± 0.0147  0.2475 ± 0.0392  0.2014 ± 0.0105
Hybrid - Weighted  0.5561 ± 0.0220  0.2407 ± 0.0162  0.2610 ± 0.0403  0.1977 ± 0.0108
Hybrid - Max       0.5566 ± 0.0226  0.2387 ± 0.0159  0.2585 ± 0.0372  0.1984 ± 0.0099

Saved: results/tabular_hybrid_runs.pkl and results/tabular_hybrid_summary.csv


## Statistical Testing — Hybrid Variants vs. Best Individual Model (Isolation Forest)

Three comparisons are made (one per fusion strategy vs. the best individual model),
forming a family of 3 tests. Bonferroni correction is applied across this family
(threshold = 0.05 / 3 = 0.0167), separate from and in addition to the model-comparison
family already tested in Notebook 04.

In [5]:
def effect_size_r(w_stat, n):
    mu = n*(n+1)/4
    sigma = np.sqrt(n*(n+1)*(2*n+1)/24)
    z = (w_stat - mu) / sigma
    return round(abs(z) / np.sqrt(n), 3)

baseline = tabular_hybrid_runs['Isolation Forest']['unsupervised']['PR-AUC']
bonferroni_alpha = 0.05 / 3

print('=== Wilcoxon signed-rank: Hybrid variant vs. Isolation Forest (baseline), PR-AUC, unsupervised condition ===\n')
for name in ['Hybrid - Average', 'Hybrid - Weighted', 'Hybrid - Max']:
    comparison = tabular_hybrid_runs[name]['unsupervised']['PR-AUC']
    stat, p = wilcoxon(baseline, comparison)
    r = effect_size_r(stat, len(baseline))
    sig = 'SIGNIFICANT' if p < bonferroni_alpha else 'not significant'
    direction = 'better' if np.mean(comparison) > np.mean(baseline) else 'worse or equal'
    print(f'{name:22s} mean PR-AUC={np.mean(comparison):.4f} vs IF={np.mean(baseline):.4f} | p={p:.4f} | r={r} | {sig} (Bonferroni α={bonferroni_alpha:.4f}) | hybrid is {direction}')

=== Wilcoxon signed-rank: Hybrid variant vs. Isolation Forest (baseline), PR-AUC, unsupervised condition ===

Hybrid - Average       mean PR-AUC=0.2326 vs IF=0.2407 | p=0.0020 | r=0.886 | SIGNIFICANT (Bonferroni α=0.0167) | hybrid is worse or equal
Hybrid - Weighted      mean PR-AUC=0.2407 vs IF=0.2407 | p=1.0000 | r=0.886 | not significant (Bonferroni α=0.0167) | hybrid is worse or equal
Hybrid - Max           mean PR-AUC=0.2387 vs IF=0.2407 | p=0.0020 | r=0.886 | SIGNIFICANT (Bonferroni α=0.0167) | hybrid is worse or equal


## Interpretation

If none of the three fusion variants significantly outperform Isolation Forest alone, this
is a valid and reportable finding under RQ2/H2 — the approval form explicitly allows for the
hybrid framework to show "improved **or complementary**" performance, and a lack of
improvement, honestly reported with proper statistical testing, is itself informative: it
indicates the Feedforward Autoencoder does not carry additional discriminative information
beyond what Isolation Forest already captures for this dataset.